## This script produces the arctic mean monthly AHT, OHU and IRF_aer

we keep various other variables in case they are useful

In [1]:
import numpy as np
import xarray as xr
import os
import glob
import matplotlib.pyplot as plt
import pandas as pd
import json
import cftime
from cftime import DatetimeNoLeap
from scipy import stats
from scipy.interpolate import interp1d
import dask
from tqdm import tqdm
from mpl_axes_aligner import align
from xmip.preprocessing import rename_cmip6
%matplotlib inline


from utils import QACC_utils
#from utils.config import model, min_lat_arctic, max_lat_arctic, min_lat_tropics, max_lat_tropics, min_year_late_cent, max_year_late_cent

##### SETTINGS
model = 'UKESM1-0-LL'
#min_lat_arctic, max_lat_arctic = 66, 90
#min_lat_tropics, max_lat_tropics = -23, 23


regions = ['Arctic', 'Tropics']
lat_band_dict = {'Arctic':[66, 90],
                 'Tropics':[-23, 23]}

scenarios = ['ssp245', 'ARISE', 'ssp245_baseline']

experiment_dict = {'ssp245':'ssp245', 
                   'ARISE':'ARISE',
                   'ssp245_baseline':'ssp245'}

time_slice_dict = {'ssp245':['2050', '2070'], 
                   'ARISE':['2050', '2070'],
                   'ssp245_baseline':['2015', '2033']}

## Part 1
presave a set of needed variables, as arctic means

In [2]:
### define set of required variables, keep more than are needed here

rad_flux_vars_surface = ['rsds', 'rsus', 'rlds', 'rlus', 'rsdscs', 'rldscs']
# all radiative vars at surface

rad_flux_vars_TOA = ['rsdt', 'rsut', 'rlut', 'rlutcs', 'rsutcs', 'rsutaf', 'rsutcsaf']
# all radiative vars at top-of-atmosphere

heat_flux_vars_surface = ['hfls', 'hfss'] 
# surface upward latent heat flux and surface upward sensible heat flux

PE_vars = ['evspsbl', 'pr']
# precip and evap vars needed to distinguish sensible and latent heat

column_stored_energy_vars = ['ta', 'hus', 'hur']

additional = ['huss'] # might need the surface humidity for q feedback

# see Donohue et al 2020 JCLIM, equation A2 https://faculty.washington.edu/karmour/papers/Donohoe_etal_JCLIM2020.pdf

AHT_vars = ['tas'] + rad_flux_vars_surface + rad_flux_vars_TOA + heat_flux_vars_surface + PE_vars + additional # + column_stored_energy_vars

In [3]:
def get_monthlyQ(ds):

    ## accounting for storage makes no difference : delta_S is < 0.001 W/m2
    
    
    # Constants
    c_p = 1004        # J/(kg·K), specific heat of dry air
    g = 9.81         # m/s², gravity
    L = 2.5e6       # J/kg, latent heat of vaporization
    seconds_per_month = 2.592e6 
    S = (1/g) * (c_p*ds.ta.sum('plev') + L*ds.hus.sum('plev')) * (1/seconds_per_month)
    delta_S = S - S.roll(month=1)

    ds['OHU'] = ds.rsds - ds.rsus + ds.rlds - ds.rlus - ds.hfls - ds.hfss
    ds['AHT'] = ds.rsut + ds.rlut - ds.rsdt + delta_S + ds['OHU']
    
    ### now split into moist and dry components. Done following Hahn et al. 2021
    ### https://www.frontiersin.org/articles/10.3389/feart.2021.710036/full
    ### we take moist energy transport as (P-E)*(latent heat of vaporisation).
    ### this neglects the latent heat of fusion for solid precip. 
    ### the dry component is then calculated as the difference between Q and moist

    L = 2.501*(10**6)# latent heat of vaporisation, assumed constant at 2.501 × 10^6 J kg−1
    ### OLD VERSION, which used evspsbl:  ###  ds['Qmoist'] = (ds.pr - ds.evspsbl)*L*unit_conversions[units]  
    ## new, uses hfls instead:
    ds['AHTmoist'] = ds.pr*L - ds.hfls 
    ds['AHTdry'] = ds['AHT'] - ds['AHTmoist']

    
    return ds


def add_extra_variables(ds, scen, min_lat):
    ## adds F, albedo, and CRE, and saves:
    # add IRF, allsky and clearsky
    ds['IRF_aer'] = ds['rsutaf'] - ds['rsut']  # sign convention here is that more SW to space is a negative forcing
    ds['IRF_aer_cs'] = ds['rsutcsaf'] - ds['rsutcs']

    # add albedo, defined as ratio of reflected up to incident down SW radiation:
    ds['albedo'] = ds['rsus']/ds['rsds']

    # add CRE (cloud radiative effect))
    ds['CRE'] = ((ds['rsutcs']-ds['rsut'])+(ds['rlutcs']-ds['rlut']))
    
    ds.to_netcdf('intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
        s=scen, l=str(min_lat), m=model))
    return ds

In [4]:
## v2, do all in one go
### preproces, a couple minutes:


for scen in scenarios:
    print(scen)
    for region in regions:
        
        print(region)
    
        # single level vars first:
        ds_sl = QACC_utils.get_all_vars_arctic_monthly(vars=AHT_vars,
                                                       model=model,
                                                       scenario=experiment_dict[scen],
                                                       min_lat=lat_band_dict[region][0],
                                                       max_lat=lat_band_dict[region][1],
                                                       min_year=time_slice_dict[scen][0], 
                                                       max_year=time_slice_dict[scen][1])
        
        ds_sl.to_netcdf('intermediate_outputs/init/{s}_{l}_{m}_{r}_singlelev.nc'.format(
            s=scen, l=str(lat_band_dict[region][0]), m=model, r=region))
    
    
        # repeat for the vars on levels
        ds_al = QACC_utils.get_all_vars_arctic_monthly(vars=column_stored_energy_vars, model=model,
                                                       scenario=experiment_dict[scen],
                                                       min_lat=lat_band_dict[region][0],
                                                       max_lat=lat_band_dict[region][1],
                                                       min_year=time_slice_dict[scen][0], 
                                                       max_year=time_slice_dict[scen][1])
        
        ds_al.to_netcdf('intermediate_outputs/init/{s}_{l}_{m}_{r}_alllevs.nc'.format(
            s=scen, l=str(lat_band_dict[region][0]), m=model, r=region))

        ds = xr.merge([ds_sl, ds_al], compat='override') # override necessary because of different number of ensemble members, which we saved as a var
        print('made combined ds')

        # add heat transport terms
        ds = get_monthlyQ(ds)
        print('added Q')

        # add final calculated vars (albedo, CRE, IRF), and save
        ds = add_extra_variables(ds, scen, lat_band_dict[region][0])
        print('done')
        

ssp245
Arctic


  0%|          | 0/19 [00:00<?, ?it/s]

tas


  5%|▌         | 1/19 [00:01<00:19,  1.08s/it]

rsds


...

hus


 67%|██████▋   | 2/3 [00:00<00:00,  2.06it/s]

hur


100%|██████████| 3/3 [00:01<00:00,  2.10it/s]


made combined ds
added Q
done


## Part 2
per-scenario ocean heat transport convergence (added in v3)

Computes the monthly-climatology OHT convergence into each latitude band from `hfbasin`
(northward ocean heat transport, W), expressed as a band-mean flux (W/m2, positive =
convergence into the band), and appends it as `OHT_conv` to the `intermediate_outputs/fin`
files written above. `04_` then splits the OHU contribution into OHT (this variable) and
OHS (residual: OHS = OHU + OHT_conv), following Hankel & Bonan (2026, GRL,
doi:10.1029/2025GL120079), who take OHT directly from the ocean model's heat-transport
diagnostic with storage as the residual.

Notes:
- global-ocean sector selected by name, not index (basin ordering not assumed)
- ensemble handling matches the Amon variables above (`read_in_ens_mean`, max 5 members)
- Arctic: convergence = +hfbasin(66N), assuming zero transport at the pole
- Tropics: convergence = hfbasin(-23) - hfbasin(+23)
- band area is land+ocean over the same `sel(y=slice())` as the band means, consistent
  with OHU being a (cos-weighted) mean over all band cells
- `hfbasin` is ocean-only: sea-ice heat transport across the gates is not included and
  therefore ends up in the OHS residual, along with any land-surface flux
- this cell is re-runnable on its own: it appends to the existing fin files without
  requiring the presave loop above to be re-executed

In [5]:
### v3 addition: OHT convergence per scenario and band, appended to fin files

gate_dict = {'Arctic':  [66.0, None],   # [southern gate, northern gate]; None = closed at the pole
             'Tropics': [-23.0, 23.0]}

# band areas (m2), land + ocean, matching the y-slice used for the band means above
areacella_path = ('/badc/cmip6/data/CMIP6/CMIP/MOHC/UKESM1-0-LL/piControl/r1i1p1f2/fx/'
                  'areacella/gn/latest/areacella_fx_UKESM1-0-LL_piControl_r1i1p1f2_gn.nc')
areacella = rename_cmip6(xr.open_dataset(areacella_path)).areacella

for scen in scenarios:
    print(scen)

    # load hfbasin (Omon); same loading + ensemble convention as the Amon vars above
    hfb = QACC_utils.read_in_ens_mean(
        QACC_utils.get_dirs('hfbasin', model, experiment_dict[scen], table='Omon'),
        zonal_mean=False, max_ens_members=5).hfbasin

    # select the global-ocean sector by name (do not trust basin index ordering)
    sectors = [str(s) for s in hfb.sector.values]
    i_global = [i for i, s in enumerate(sectors) if 'global' in s.lower()]
    assert len(i_global) == 1, 'could not uniquely identify global sector in: {}'.format(sectors)
    hfb = hfb.isel(basin=i_global[0])
    print('  selected sector:', sectors[i_global[0]])

    # monthly climatology over the same window as the other variables
    hfb = hfb.sel(time=slice(time_slice_dict[scen][0],
                             time_slice_dict[scen][1])).groupby('time.month').mean('time').load()

    for region in regions:
        min_lat, max_lat = lat_band_dict[region]
        A_band = float(areacella.sel(y=slice(min_lat, max_lat)).sum())  # m2, land + ocean

        south_gate, north_gate = gate_dict[region]
        conv_W = hfb.interp(y=south_gate)                # northward OHT in at the southern gate
        if north_gate is not None:
            conv_W = conv_W - hfb.interp(y=north_gate)   # minus northward OHT out at the northern gate

        OHT_conv = (conv_W / A_band).rename('OHT_conv')
        OHT_conv = OHT_conv.drop_vars([c for c in OHT_conv.coords if c != 'month'])
        OHT_conv.attrs = {
            'units': 'W m-2',
            'long_name': 'ocean heat transport convergence into the {} band, band-mean'.format(region),
            'sign_convention': 'positive = convergence into the band',
            'gates': str(gate_dict[region]),
            'source': 'hfbasin (Omon), global-ocean sector, ensemble mean (max 5 members)',
            'note': ('ocean-only: excludes sea-ice heat transport across the gates. '
                     'OHS is derived in 04_ as the residual OHU + OHT_conv.')}

        # append to the existing fin file (load fully, close, overwrite in place)
        fin_path = 'intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
            s=scen, l=str(min_lat), m=model)
        ds_fin = xr.open_dataset(fin_path).load()
        ds_fin.close()
        assert (ds_fin.month.values == OHT_conv.month.values).all(), 'month coord mismatch'
        ds_fin['OHT_conv'] = OHT_conv
        ds_fin.to_netcdf(fin_path)

        print('  {r}: annual-mean {pw:+.3f} PW = {wm2:+.2f} W/m2  ->  {p}'.format(
            r=region, pw=float(conv_W.mean()) / 1e15,
            wm2=float(OHT_conv.mean()), p=fin_path))


ssp245
  selected sector: b'global_ocean'
  Arctic: annual-mean +0.324 PW = +14.99 W/m2  ->  intermediate_outputs/fin/ssp245_66_UKESM1-0-LL.nc
  Tropics: annual-mean -2.045 PW = -10.47 W/m2  ->  intermediate_outputs/fin/ssp245_-23_UKESM1-0-LL.nc
ARISE
  selected sector: b'global_ocean'
  Arctic: annual-mean +0.291 PW = +13.46 W/m2  ->  intermediate_outputs/fin/ARISE_66_UKESM1-0-LL.nc
  Tropics: annual-mean -2.173 PW = -11.13 W/m2  ->  intermediate_outputs/fin/ARISE_-23_UKESM1-0-LL.nc
ssp245_baseline
  selected sector: b'global_ocean'
  Arctic: annual-mean +0.292 PW = +13.52 W/m2  ->  intermediate_outputs/fin/ssp245_baseline_66_UKESM1-0-LL.nc
  Tropics: annual-mean -2.352 PW = -12.05 W/m2  ->  intermediate_outputs/fin/ssp245_baseline_-23_UKESM1-0-LL.nc
